In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, sum as _sum, avg

spark = SparkSession.builder \
    .appName("Tugas4") \
    .master("local[*]") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/10 04:12:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/10 04:12:39 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
# 1. Membaca dan Eksplorasi Awal
path_hdfs = "hdfs://localhost:9000/user/zerouno/tugas4/transaksi_september_2026.csv"
df = spark.read.csv(path_hdfs, header=True, inferSchema=True)

df.printSchema()
print("Jumlah baris:", df.count())
df.show(10)

root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)

Jumlah baris: 1000
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|        

In [3]:
# 2. Menangani Data Kosong
null_count = df.filter(col("rating").isNull()).count()
print("Jumlah data kosong pada kolom rating:", null_count)
df_clean = df.na.fill({"rating": 0})
df_clean.show(5)

Jumlah data kosong pada kolom rating: 204
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|           8|       60000|         E-Wallet|   3.0|
|ORD-3003|2026-09-09 00:00:00|   Makanan & Minuman|  Semarang|           6|      350000|    Transfer Bank|   4.0|
|ORD-3004|2026-09-10 00:00:00|        Rumah Tangga|Yogyakarta|          10|       60000|         E-Wallet|   4.0|
+--------+-------------------+----------------

*Penjelasan*
Metode df.na.fill() dipilih agar data transaksi yang tidak memiliki rating tetap tersimpan dan tidak terbuang. Jika menggunakan df.na.drop(), baris data transaksi tersebut akan terhapus sehingga dapat mengurangi jumlah total sampel transaksi dan memengaruhi akurasi analisis agregasi pendapatan atau volume transaksi.

In [4]:
# 3. Transformasi Data
df_transformed = df_clean.withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan")) \
                         .withColumn("tier_transaksi", when(col("total_pendapatan") > 500000, "Besar").otherwise("Kecil"))

df_transformed.show(5)

+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|total_pendapatan|tier_transaksi|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|          270000|         Kecil|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|          600000|         Besar|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|           8|       60000|         E-Wallet|   3.0|          480000|         Kecil|
|ORD-3003|2026-09-09 00:00:00|   Makanan & Minuman|  Semarang|           6|      350000|    Transfer Bank|   4.0|         21

In [5]:
# 4. Analisis dengan GroupBy
# A. Kategori apa yang memiliki total_pendapatan tertinggi?
df_transformed.groupBy("kategori") \
    .agg(_sum("total_pendapatan").alias("total_pendapatan")) \
    .orderBy(col("total_pendapatan").desc()) \
    .limit(1) \
    .show()

# B. Kota mana dengan jumlah transaksi tier "Besar" terbanyak?
df_transformed.filter(col("tier_transaksi") == "Besar") \
    .groupBy("kota") \
    .count() \
    .orderBy(col("count").desc()) \
    .limit(1) \
    .show()

# C. Berapa rata-rata rating untuk masing-masing metode_pembayaran?
df_transformed.groupBy("metode_pembayaran") \
    .agg(avg("rating").alias("rata_rata_rating")) \
    .show()

+------------+----------------+
|    kategori|total_pendapatan|
+------------+----------------+
|Rumah Tangga|       138665000|
+------------+----------------+

+----+-----+
|kota|count|
+----+-----+
|Solo|   92|
+----+-----+

+-----------------+------------------+
|metode_pembayaran|  rata_rata_rating|
+-----------------+------------------+
|              COD|3.3745019920318726|
|    Transfer Bank|3.3399209486166006|
|     Kartu Kredit|3.1910569105691056|
|         E-Wallet|             3.292|
+-----------------+------------------+



In [7]:
# 5. Menyimpan Hasil ke HDFS
output_hdfs = "hdfs://localhost:9000/user/zerouno/tugas4/hasil_transaksi_september_2026"

# Menyimpan ke HDFS dalam format CSV
df_transformed.write.csv(output_hdfs, header=True, mode="overwrite")

# Verifikasi keberhasilan dengan membaca kembali data dari HDFS
df_verify = spark.read.csv(output_hdfs, header=True, inferSchema=True)
df_verify.show(5, truncate=False)
print("Verifikasi berhasil. Jumlah baris data tersimpan:", df_verify.count())

+--------+-------------------+----------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|order_id|tanggal            |kategori              |kota      |unit_terjual|harga_satuan|metode_pembayaran|rating|total_pendapatan|tier_transaksi|
+--------+-------------------+----------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|ORD-3000|2026-09-02 00:00:00|Rumah Tangga          |Yogyakarta|3           |90000       |COD              |4.0   |270000          |Kecil         |
|ORD-3001|2026-09-04 00:00:00|Makanan & Minuman     |Solo      |3           |200000      |E-Wallet         |5.0   |600000          |Besar         |
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecantikan|Semarang  |8           |60000       |E-Wallet         |3.0   |480000          |Kecil         |
|ORD-3003|2026-09-09 00:00:00|Makanan & Minuman     |Semarang  |6           |350000      |Transfer Bank    |4.0 